# Module 2 — SQL Server 2025 as a Vector Store

### *AI for the Rest of Us* · Day of Data BR Pre-Con

In Module 1 you saw that an **Embedding** turns text into numbers, and that similar meanings land close together. Now we put that to work **inside your database**.

**The big idea:** SQL Server 2025 has a native **`VECTOR`** column type and a **`VECTOR_DISTANCE`** function. That means your ordinary SQL Server *is* a **VectorStore** — a place that holds Embeddings and can rank rows by closeness in meaning. No separate 'vector database' product required.

You'll run a real **SemanticSearch** against 40 live Products and watch meaning beat keywords.

## Vocabulary (from the workshop glossary)

- **Product** — one row in our catalog: Name, Description, Category, Price, and its Embedding.
- **Catalog** — the `WorkshopCatalog` database of Products in SQL Server 2025.
- **Embedding** — the 384 numbers a model makes from a Product's Description.
- **Vector** — the SQL Server 2025 `VECTOR(384)` **column** that stores an Embedding.
- **VectorStore** — a place that holds Vectors and ranks them by closeness. Here it's SQL Server itself.
- **SemanticSearch** — finding Products by *meaning*: embed the query, rank by `VECTOR_DISTANCE`.
- **keyword search** — finding rows by *exact words* (`LIKE '%...%'`). The thing we'll beat.

## Step 0 — Load your data first  *(run once, in a Terminal)*

Before the cells below can search anything, your **Catalog** needs Products and their Embeddings. From the repo
folder (with your `.venv` active), run these **commands in a Terminal** — not in this notebook:

```bash
python db/setup_catalog.py      # create WorkshopCatalog + load 70 Products
python db/populate_catalog.py   # embed each Description into the VECTOR(384) column
python db/create_readonly_login.py  # create the read-only login your Module-5 tool will use
```

> **Backup (no Docker):** if your SQL Server container won't start, run `python db/setup_sqlite.py` (it loads *and*
> embeds in one step), then `export WORKSHOP_DB=sqlite` (Windows: `set WORKSHOP_DB=sqlite`) **before you launch Jupyter**.
> The notebook auto-detects it and ranks with the same cosine math in Python — **same results**, nothing else to change.

Done? The next cell connects and should report **70 Products, 70 with an Embedding**. Then run the rest top to bottom.

In [ ]:
# Setup: connect to the Catalog and load the same embedding helpers from Module 1.
import warnings
warnings.filterwarnings("ignore")

import os, sys
PROJECT_ROOT = os.path.dirname(os.getcwd())  # modules/ -> project root
sys.path.insert(0, os.path.join(PROJECT_ROOT, "db"))
from embeddings import default_embedder, vector_literal, cosine_distance  # noqa: E402

# Which backend? SQL Server 2025 by default, or the SQLite fallback if you set WORKSHOP_DB=sqlite.
BACKEND = os.environ.get("WORKSHOP_DB", "mssql").lower()

if BACKEND == "sqlite":
    import sqlite3
    conn = sqlite3.connect(os.path.join(PROJECT_ROOT, "db", "workshop_catalog.sqlite"))
else:
    import mssql_python
    conn = mssql_python.connect(
        "Server=localhost,1433;Database=WorkshopCatalog;"
        "UID=sa;PWD=Workshop!2026Pass;Encrypt=yes;TrustServerCertificate=yes;"
    )

cur = conn.cursor()
cur.execute("SELECT COUNT(*), COUNT(DescriptionEmbedding) FROM Products")
total, embedded = cur.fetchone()
print(f"Connected to the {BACKEND} Catalog. {total} Products, {embedded} with an Embedding stored.")

## A peek at the Vector column

The `DescriptionEmbedding` column is a real SQL Server **`VECTOR(384)`** — the same 384 numbers you made in Module 1, now living in a table. Let's look at one Product and the start of its Vector.

In [ ]:
# Peek at one stored Embedding. It's long (384 numbers), so show just the start.
if BACKEND == "sqlite":
    # SQLite stores the Embedding as JSON text -- read it directly.
    cur.execute("SELECT Name, Category, Price, DescriptionEmbedding "
                "FROM Products ORDER BY ProductID LIMIT 1")
else:
    # SQL Server stores a VECTOR -- cast it to text so we can eyeball it.
    cur.execute("SELECT TOP (1) Name, Category, Price, "
                "CAST(DescriptionEmbedding AS NVARCHAR(MAX)) "
                "FROM Products ORDER BY ProductID")
name, category, price, vector_text = cur.fetchone()
print("Product :", name, f"({category}, ${price})")
print("Vector  :", vector_text[:70], "...")
print()
print("That column holds the meaning of the Description as 384 numbers.")

## SemanticSearch: rank Products by meaning

Here's the payoff. We take a plain-English query, embed it with the **same model** from Module 1, and hand those numbers to SQL Server. The database ranks every Product by `VECTOR_DISTANCE('cosine', ...)` — **smaller distance = closer in meaning** — and returns the top 5.

The query we send shares almost no words with the products it will find. Meaning does the work.

```sql
SELECT TOP (?) Name, Category, Price,
       VECTOR_DISTANCE('cosine', DescriptionEmbedding, CAST(? AS VECTOR(384))) AS distance
FROM Products
ORDER BY distance;
```

In [ ]:
import json

# 1) The query, in plain English (no product keywords on purpose):
query_text = "a waterproof jacket for cold hikes"

# SQL Server 2025 ranks natively with VECTOR_DISTANCE. The SQLite backup has no VECTOR type,
# so it computes the SAME cosine distance in Python -- the ranking comes out identical.
semantic_sql = (
    "SELECT TOP (?) Name, Category, Price, "
    "Description AS why_matched, "
    "CAST(VECTOR_DISTANCE('cosine', DescriptionEmbedding, "
    "CAST(? AS VECTOR(384))) AS DECIMAL(9,6)) AS distance "
    "FROM Products ORDER BY distance"
)

def show_semantic(text, k=5):
    """Embed the text, rank Products by meaning, and print each with WHY it matched."""
    query_vector = default_embedder(text)
    if BACKEND == "sqlite":
        cur.execute("SELECT Name, Category, Price, Description, DescriptionEmbedding FROM Products")
        ranked = sorted(
            ((n, c, p, why, cosine_distance(query_vector, json.loads(emb)))
             for n, c, p, why, emb in cur.fetchall()),
            key=lambda row: row[4],
        )
        rows = ranked[:k]
    else:
        cur.execute(semantic_sql, [k, vector_literal(query_vector)])
        rows = cur.fetchall()
    print(f"SemanticSearch for: {text!r}\n")
    for i, (name, category, price, why, distance) in enumerate(rows, 1):
        print(f"{i}. {float(distance):.3f}  {name}  ({category}, ${price})")
        print(f"       why: {why}")
    print()

show_semantic(query_text, k=5)
print("Top hits are rain / cold-weather gear -- even though 'waterproof' never appears in them.")
print("Read the 'why:' line -- that's the Description we searched. It matched on MEANING, not words.")

## Now the contrast: keyword search

Let's ask the *old* way — an exact-word `LIKE` search for **"waterproof"**. Same intent, but it only finds rows that literally contain that word. Watch what happens.

In [ ]:
keyword = "waterproof"
cur.execute(
    "SELECT Name, Category, Price FROM Products "
    "WHERE Description LIKE ? OR Name LIKE ?",
    [f"%{keyword}%", f"%{keyword}%"],
)
rows = cur.fetchall()

print(f"Keyword search for {keyword!r} (exact word match):\n")
if rows:
    for name, category, price in rows:
        print(f"  {name} ({category}, ${price})")
else:
    print("  (no matches -- not one Product literally says 'waterproof')")
print()
print("Keyword search finds nothing. SemanticSearch found the rain trench and the parka.")
print("Same data, new superpower: your database now searches by MEANING.")

---
## Activity: your turn

Change the query and see meaning win again. In the cell below:

1. Replace `my_query` with something **in your own words** — describe a product by what it's *for*, not by its name.
2. Run it. Do the top hits make sense?
3. **Bonus:** pick a query that shares **zero words** with the product you expect to see (e.g. *"something to keep my coffee hot on the drive to work"*). Semantic search should still find it.

In [ ]:
# YOUR TURN: change this line to anything you'd ask, then run the cell.
#   Try: "a gift for someone who loves to cook"  /  "a way to listen to music on the go"
#        "keep my desk tidy and organized"       /  "gear for camping in the mountains"
my_query = "a way to listen to music on the go"

show_semantic(my_query, k=5)

## Wrap-up

- SQL Server 2025's **`VECTOR`** column + **`VECTOR_DISTANCE`** make your database a **VectorStore**.
- **SemanticSearch** embeds the query and ranks Products by meaning — it found rain gear for *"waterproof jacket"* when a keyword search found nothing.
- It's pure T-SQL: no separate vector database, no application layer required. *(Your Instructor will show the very same search in SSMS — see `db/ssms_demo.sql`.)*

**Next up:** we let an AI call this search *for you* over MCP — plain-English questions, real answers from your data.

*(Housekeeping: the cell below closes the database connection.)*

In [ ]:
conn.close()
print("Connection closed. Nice work -- you turned SQL Server into a semantic search engine.")